# Fine-tune a Small LLM (Notebook Version)

This notebook is the Jupyter equivalent of `train.py`.
It loads blended JSONL data, trains a LoRA adapter with **Unsloth** when available, and falls back to **Transformers + PEFT** otherwise.


## 1) Configuration
Adjust these values before running training cells.


In [ ]:
from pathlib import Path
import json

# Core config (matches train.py defaults)
BASE_MODEL = "unsloth/Qwen2.5-0.5B-bnb-4bit"
DATA_DIR = "data"
OUTPUT_DIR = "output"

LORA_RANK = 16
LORA_ALPHA = 32
MAX_SEQ_LENGTH = 256
EPOCHS = 3
MAX_STEPS = -1  # set >0 for smoke tests
BATCH_SIZE = 2
GRAD_ACCUM = 8
LR = 2e-4
WARMUP_STEPS = 10
LOGGING_STEPS = 5
SAVE_STEPS = 50
FP16 = False
BF16 = True

SEED = 42


## 2) Helpers + data loading


In [ ]:
def load_jsonl(path: str) -> list[dict]:
    data = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

train_path = Path(DATA_DIR) / "train.jsonl"
val_path = Path(DATA_DIR) / "val.jsonl"

if not train_path.exists():
    raise FileNotFoundError(f"{train_path} not found. Run `python prepare_data.py` first.")

train_data = load_jsonl(str(train_path))
val_data = load_jsonl(str(val_path)) if val_path.exists() else []

print(f"Train samples: {len(train_data)}")
print(f"Val samples:   {len(val_data)}")


## 3) Training (Unsloth first, PEFT fallback)


In [ ]:
from datasets import Dataset

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

try:
    from unsloth import FastLanguageModel

    print(f"Loading base model with Unsloth: {BASE_MODEL}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        bias="none",
        use_gradient_checkpointing="unsloth",
    )

    trainable, total = model.get_nb_trainable_parameters()
    print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

    def formatting_func(examples):
        texts = []
        for text in examples["text"]:
            texts.append(text + tokenizer.eos_token)
        return {"text": texts}

    train_dataset = Dataset.from_list([{"text": s["text"]} for s in train_data])
    val_dataset = Dataset.from_list([{"text": s["text"]} for s in val_data]) if val_data else None

    train_dataset = train_dataset.map(formatting_func, batched=True, remove_columns=["text"])
    if val_dataset:
        val_dataset = val_dataset.map(formatting_func, batched=True, remove_columns=["text"])

    from trl import SFTTrainer
    from transformers import TrainingArguments

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=EPOCHS if MAX_STEPS <= 0 else 1,
        max_steps=MAX_STEPS if MAX_STEPS > 0 else -1,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        warmup_steps=WARMUP_STEPS,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        save_total_limit=3,
        fp16=FP16,
        bf16=BF16 and not FP16,
        evaluation_strategy="steps" if val_dataset else "no",
        eval_steps=SAVE_STEPS if val_dataset else None,
        report_to="none",
        seed=SEED,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        args=training_args,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=True,
    )

    print("\n=== Starting training (Unsloth) ===")
    trainer.train()

    final_path = output_dir / "final"
    model.save_pretrained(str(final_path))
    tokenizer.save_pretrained(str(final_path))

    merged_path = output_dir / "merged_16bit"
    model.save_pretrained_merged(str(merged_path), tokenizer, save_method="merged_16bit")

    print(f"Model saved to {final_path}")
    print(f"Merged model saved to {merged_path}")

except ImportError:
    print("Unsloth not available; falling back to Transformers + PEFT.")

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
    from peft import LoraConfig, get_peft_model, TaskType
    from trl import SFTTrainer

    model_name = BASE_MODEL
    if model_name.startswith("unsloth/"):
        hf_name = model_name.replace("unsloth/", "")
        for suffix in ["-bnb-4bit", "-bnb-8bit"]:
            hf_name = hf_name.replace(suffix, "")
        hf_mapping = {
            "Qwen2.5-0.5B": "Qwen/Qwen2.5-0.5B",
            "Qwen2.5-1.5B": "Qwen/Qwen2.5-1.5B",
            "Qwen2.5-3B": "Qwen/Qwen2.5-3B",
        }
        model_name = hf_mapping.get(hf_name, f"Qwen/{hf_name}")

    print(f"Loading fallback model: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    load_kwargs = {"trust_remote_code": True}
    if torch.cuda.is_available():
        from transformers import BitsAndBytesConfig
        load_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        load_kwargs["device_map"] = "auto"
    else:
        load_kwargs["torch_dtype"] = torch.float32

    model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        bias="none",
    )
    model = get_peft_model(model, lora_config)

    def tokenize(examples):
        return tokenizer(
            [t + tokenizer.eos_token for t in examples["text"]],
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            padding="max_length",
        )

    train_dataset = Dataset.from_list([{"text": s["text"]} for s in train_data])
    val_dataset = Dataset.from_list([{"text": s["text"]} for s in val_data]) if val_data else None

    train_dataset = train_dataset.map(tokenize, batched=True, remove_columns=["text"])
    if val_dataset:
        val_dataset = val_dataset.map(tokenize, batched=True, remove_columns=["text"])

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=EPOCHS if MAX_STEPS <= 0 else 1,
        max_steps=MAX_STEPS if MAX_STEPS > 0 else -1,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        warmup_steps=WARMUP_STEPS,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        save_total_limit=3,
        fp16=FP16 and torch.cuda.is_available(),
        bf16=BF16 and not FP16 and torch.cuda.is_available(),
        evaluation_strategy="steps" if val_dataset else "no",
        eval_steps=SAVE_STEPS if val_dataset else None,
        report_to="none",
        seed=SEED,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        args=training_args,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=True,
    )

    print("\n=== Starting training (PEFT fallback) ===")
    trainer.train()

    final_path = output_dir / "final"
    model.save_pretrained(str(final_path))
    tokenizer.save_pretrained(str(final_path))
    print(f"Model saved to {final_path}")


## Notes
- This notebook mirrors the behavior of `train.py`.
- For a quick smoke test, set `MAX_STEPS = 60`.
- Make sure `data/train.jsonl` exists (run `python prepare_data.py` first).
